# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [2]:
import pandas as pd
import numpy as np


url1 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
url2 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv"
url3 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"

df1 = pd.read_csv(url1)
df2 = pd.read_csv(url2)
df3 = pd.read_csv(url3)

df = pd.concat([df1, df2, df3])

In [3]:
# convert the Missing value from the customer tp nan and drop from tathat subset all the rows
# 1. Convert the literal text to actual NaN (Not a Number) objects
df["Customer"] = df["Customer"].replace("Missing values", np.nan)
# droping all the rows were is missing customer ID
df = df.dropna(subset=["Customer"])
df.drop_duplicates(subset=["Customer"])

,Customer,ST,GENDER,Education,Customer Lifetime Value,Income,Monthly Premium Auto,Number of Open Complaints,Policy Type,Vehicle Class,Total Claim Amount,State,Gender
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934,NaN,NaN
1,QZ44356,Arizona,F,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935,NaN,NaN
2,AI49188,Nevada,F,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247,NaN,NaN
3,WW63253,California,M,Bachelor,764586.18%,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344,NaN,NaN
4,GA49547,Washington,M,High School or Below,536307.65%,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7065,LA72316,NaN,NaN,Bachelor,23405.98798,71941.0,73.0,0,Personal Auto,Four-Door Car,198.234764,California,M
7066,PK87824,NaN,NaN,College,3096.511217,21604.0,79.0,0,Corporate Auto,Four-Door Car,379.200000,California,F
7067,TD14365,NaN,NaN,Bachelor,8163.890428,0.0,85.0,3,Corporate Auto,Four-Door Car,790.784983,California,M
7068,UP19263,NaN,NaN,College,7524.442436,21941.0,96.0,0,Personal Auto,Four-Door Car,691.200000,California,M


In [4]:
# the last 2 columns has the missing values from the ST and Gender
df["ST"] = df["ST"].fillna(df["State"])
df["GENDER"] = df["GENDER"].fillna(df["Gender"])

# Dropping the last 2 columns
df = df.drop(["State", "Gender"], axis=1)

In [5]:
# rename the columns
df.columns = [
    "customer",
    "state",
    "gender",
    "education",
    "customer_lifetime_value",
    "income",
    "monthley_premium_auto",
    "number_of_open_complaints",
    "policy_type",
    "vehicle_class",
    "total_claim_amount",
]

In [6]:
# Correcting the gender column
maping_gender = {"Femal": "F", "female": "F", "Male": "M"}
df["gender"] = df["gender"].replace(maping_gender, regex=True)

# Correcting the state column
maping_state = {
    "Cali": "California",
    "AZ": "Arizona",
    "WA": "Washington",
    "Californiafornia": "California",
}

df["state"] = df["state"].replace(maping_state)


df["education"] = df["education"].replace("Bachelors", "Bachelor")

df["customer_lifetime_value"] = df["customer_lifetime_value"].replace(
    "%", "", regex=True
)  # Use regex or you can't remove the %

maping_vehicle_calss = {"Sports Car": "Luxury", "Luxury Car": "Luxury", "Luxury SUV": "Luxury"}
df["vehicle_class"] = df["vehicle_class"].replace(maping_vehicle_calss)
df_unique = df["vehicle_class"].unique()

In [7]:
# Change the type for the customer_lifetime_value
df["customer_lifetime_value"] = pd.to_numeric(df["customer_lifetime_value"], errors="coerce")

# split and take the second value
df["number_of_open_complaints"] = df["number_of_open_complaints"].apply(
    lambda x: x.split("/")[1] if isinstance(x, str) else x
)

# Convert to numeric, turning any errors or remaining weird text into NaN
df["number_of_open_complaints"] = pd.to_numeric(df["number_of_open_complaints"], errors="coerce")

In [8]:
# Change the type for the customer_lifetime_value
df["customer_lifetime_value"] = pd.to_numeric(df["customer_lifetime_value"], errors="coerce")

# split and take the second value
df["number_of_open_complaints"] = df["number_of_open_complaints"].apply(
    lambda x: x.split("/")[1] if isinstance(x, str) else x
)

# Convert to numeric, turning any errors or remaining weird text into NaN
df["number_of_open_complaints"] = pd.to_numeric(df["number_of_open_complaints"], errors="coerce")
df.dtypes

customer                      object
state                         object
gender                        object
education                     object
customer_lifetime_value      float64
income                       float64
monthley_premium_auto        float64
number_of_open_complaints      int64
policy_type                   object
vehicle_class                 object
total_claim_amount           float64
dtype: object

In [9]:
# Drop were we don't have any values

# 1. Convert the literal text to actual NaN (Not a Number) objects
df = df.replace("Missing value", np.nan)
# 2. Now you can use dropna() effectively.
# Using subset=['Customer'] ensures we only drop the rows where the ID is gone.
df = df.dropna(subset=["customer"])

# calcualte the mode for the customer_lifetime_value excluding 0.0, in case there are multiple take the first one [0]
mode_val = df["customer_lifetime_value"][df["customer_lifetime_value"] != 0.0].mode()[0]
# Replace 0.0 with the mode
df["customer_lifetime_value"] = df["customer_lifetime_value"].replace(0.0, mode_val)
# Fill the Na with mode
df["customer_lifetime_value"] = df["customer_lifetime_value"].fillna(mode_val)

# For income calcualte the mode and replace 0.0 with the mode
mode_val = df["income"][df["income"] != 0.0].mode()[0]
df["income"] = df["income"].replace(0.0, mode_val)

#  Convert the literal text to actual NaN (Not a Number) objects
df["gender"] = df["gender"].replace("Missing value", np.nan)
# For gender replace Missing value with the avobe value
df["gender"] = df["gender"].fillna("F")

In [10]:
# No duplicate
df.duplicated()
df.reset_index(inplace=True)
df.to_csv("cleaned_data.csv", index=False)

# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [ ]:
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"

df_ch2 = pd.read_csv(url)

department_sales = round(
    df_ch2.pivot_table(index="sales_channel", values="total_claim_amount", aggfunc="sum"), 2
)
print(f"{department_sales}\n\n")
# The sales chanel "Agent" had more revenue

avg_cust = round(
    df_ch2.pivot_table(
        index=["education", "gender"], values="customer_lifetime_value", aggfunc="mean"
    ),
    2,
).sort_values("customer_lifetime_value", ascending=False)
print(avg_cust)
# The highest customer value is the F with the High school or below education lvl and the lowes is Male gender with Doctor degree

               total_claim_amount
sales_channel                    
Agent                  1810226.82
Branch                 1301204.00
Call Center             926600.82
Web                     706600.04


                             customer_lifetime_value
education            gender                         
High School or Below F                       8675.22
Master               M                       8168.83
                     F                       8157.05
High School or Below M                       8149.69
College              M                       8052.46
Bachelor             F                       7874.27
College              F                       7748.82
Bachelor             M                       7703.60
Doctor               M                       7415.33
                     F                       7328.51


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*